# Initialising Edges, tweets, users metadata Files 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

df_edges = pd.read_parquet("datasets/df_edges_final.parquet")
df_user_tweets = pd.read_parquet("datasets/df_user_tweets_final.parquet")
df_user = pd.read_parquet("datasets/df_users_final.parquet")
df_list = pd.read_csv("datasets/list.csv")
df_hashtag = pd.read_json("datasets/hashtag.json")

In [2]:
print(df_edges.head())
print(df_user.head())
print(df_user_tweets.head())
print(df_list.head())
print(df_hashtag.head())

              source_id             target_id   relation
0            u148520716             u59653593  following
1  u1078324065532764166   u781676758932262912  following
2             u33746386             u22730752  following
3             u28896955           u1951553371  following
4           u3287999704  u1029392984842719233  followers
                     id  label  split  participation_degree                 created_at                                        description                 location            name  pinned_tweet_id  \
0  u1217628182611927040  human   test                  1565  2020-01-16 02:02:55+00:00  Theoretical Computer Scientist. See also https...            Cambridge, MA      Boaz Barak              NaN   
1           u1679822588    bot  train                  1097  2013-08-18 04:21:48+00:00                                                                          🇬🇧           Grian     1.143808e+18   
2           u1519144464  human  train                  1066  

Prep cells

In [3]:
def safe_list(x):
    return x if isinstance(x, list) else []

def norm_id(x):
    if pd.isna(x):
        return None
    return str(x)

def get_nested(obj, *keys, default=None):
    cur = obj
    for k in keys:
        if isinstance(cur, dict) and k in cur:
            cur = cur[k]
        else:
            return default
    return cur

# Keep ids as strings
df_edges = df_edges.copy()
df_user = df_user.copy()
df_user_tweets = df_user_tweets.copy()
df_list = df_list.copy()

df_edges["source_id"] = df_edges["source_id"].astype(str)
df_edges["target_id"] = df_edges["target_id"].astype(str)
df_user["id"] = df_user["id"].astype(str)
df_user_tweets["id"] = df_user_tweets["id"].astype(str)

valid_users = set(df_user["id"])

# Split the existing edge table by relation

df_following_base = df_edges[df_edges["relation"] == "following"].copy()
df_follower_base = df_edges[df_edges["relation"] == "followed"].copy()
df_post_base      = df_edges[df_edges["relation"] == "post"].copy()
df_like_base      = df_edges[df_edges["relation"] == "like"].copy()
df_own_base     = df_edges[df_edges["relation"] == "own"].copy()
df_contain_base = df_edges[df_edges["relation"] == "contain"].copy()
df_mentions_base = df_edges[df_edges["relation"] == "mentioned"].copy()
df_replies_base  = df_edges[df_edges["relation"] == "replied_to"].copy()
df_retweets_base = df_edges[df_edges["relation"] == "retweeted"].copy()
df_quotes_base   = df_edges[df_edges["relation"] == "quoted"].copy()
df_pinned_base = df_edges[df_edges["relation"] == "pinned"].copy()
df_membership_base = df_edges[df_edges["relation"] == "membership"].copy()
df_discuss_base = df_edges[df_edges["relation"] == "discuss"].copy()

print("following:", df_following_base.shape)
print("follower:", df_follower_base.shape)
print("post:", df_post_base.shape)
print("like:", df_like_base.shape)
print("own:", df_own_base.shape)
print("contain:", df_contain_base.shape)
print("mentions:", df_mentions_base.shape)
print("replies_to:", df_replies_base.shape)
print("retweets:", df_retweets_base.shape)
print("quotes:", df_quotes_base.shape)
print("pinned:", df_pinned_base.shape)
print("membership:", df_membership_base.shape)
print("discuss:", df_discuss_base.shape)


following: (197990, 3)
follower: (17346, 3)
post: (5797761, 3)
like: (203642, 3)
own: (2874, 3)
contain: (1562941, 3)
mentions: (849521, 3)
replies_to: (489842, 3)
retweets: (740414, 3)
quotes: (135434, 3)
pinned: (9120, 3)
membership: (102436, 3)
discuss: (4970315, 3)


# Build mapping from "user -> tweet -> user" for easy authorship retrieval

In [4]:
tweet_to_user = (
    df_post_base[["source_id", "target_id"]]
    .drop_duplicates()
    .rename(columns={"source_id": "author_user_id", "target_id": "tweet_id"})
)

tweet_to_user["author_user_id"] = tweet_to_user["author_user_id"].astype(str)
tweet_to_user["tweet_id"] = tweet_to_user["tweet_id"].astype(str)

tweet_to_user_map = dict(zip(tweet_to_user["tweet_id"], tweet_to_user["author_user_id"]))

print("tweet_to_user from post edges:", len(tweet_to_user_map))
tweet_to_user.head()

tweet_to_user from post edges: 5797761


,author_user_id,tweet_id
242793,u48390963,t1480809188880175108
242794,u368644797,t1455239713267363849
242795,u904794157,t1499039530996879361
242796,u493482998,t1488474786149847041
242797,u2882501298,t1491697289026486277


# 1) User --follows -> user

In [5]:
df_followers_flipped = (
    df_follower_base[["source_id", "target_id"]]
    .rename(columns={"source_id": "target_id", "target_id": "source_id"})
    [["source_id", "target_id"]]
    .copy()
)

df_following_all = pd.concat([
    df_following_base[["source_id", "target_id"]].copy(),
    df_followers_flipped
], ignore_index=True)

df_following_all["source_id"] = df_following_all["source_id"].astype(str)
df_following_all["target_id"] = df_following_all["target_id"].astype(str)
df_following_all["relation"] = "following"

df_follows = df_following_all[
    df_following_all["source_id"] != df_following_all["target_id"]
][["source_id", "target_id", "relation"]].drop_duplicates()

print(df_follows.shape)
df_follows.head()

(215336, 3)


,source_id,target_id,relation
0,u148520716,u59653593,following
1,u1078324065532764166,u781676758932262912,following
2,u33746386,u22730752,following
3,u28896955,u1951553371,following
4,u625655093,u141163282,following


# 2) user --likes -> user

In [6]:
df_likes_user = (
    df_like_base[["source_id", "target_id"]]
    .rename(columns={"source_id": "source_id", "target_id": "tweet_id"})
    .assign(target_id=lambda x: x["tweet_id"].astype(str).map(tweet_to_user_map))
    .dropna(subset=["target_id"])
)
df_likes_user["relation"] = "likes"

df_likes_user = df_likes_user[
    df_likes_user["source_id"] != df_likes_user["target_id"]
][["source_id", "target_id", "relation"]].drop_duplicates()

print(df_likes_user.shape)
df_likes_user.head()

(22913, 3)


,source_id,target_id,relation
13423336,u2248919588,u51310666,likes
13423343,u1111600251939377152,u848081553368416256,likes
13423344,u287156977,u18839785,likes
13423361,u56780944,u816653,likes
13423362,u72448544,u1036997113274662920,likes


# 3)  user --owner_of_communities_frequent_by -> user

In [7]:
# own = user -> list
df_own_links = df_own_base[["source_id", "target_id"]].copy()
df_own_links.columns = ["owner_user_id", "list_id"]

#contain = list -> tweet
df_contain_links = df_contain_base[["source_id", "target_id"]].copy()
df_contain_links.columns = ["list_id", "tweet_id"]

# post = user -> list -> tweet -> user
df_post_links = df_post_base[["source_id", "target_id"]].copy()
df_post_links.columns = ["author_user_id", "tweet_id"]

# user -> tweet ->
df_list_owner_to_user = (
    df_own_links
    .merge(df_contain_links, on="list_id", how="inner")
    .merge(df_post_links, on="tweet_id", how="inner")
)

#Drop any self loops
df_list_owner_to_user = df_list_owner_to_user[
    (df_list_owner_to_user["owner_user_id"] != df_list_owner_to_user["author_user_id"])
].copy()

# final format (user - user)
df_list_owner_to_user = df_list_owner_to_user.rename(columns={
    "owner_user_id": "source_id",
    "author_user_id": "target_id"
})[["source_id", "target_id"]]

df_list_owner_to_user["relation"] = "owner_of_communities_frequent_by"

print(df_list_owner_to_user.shape)
df_list_owner_to_user.head()

(43159, 3)


,source_id,target_id,relation
3,u20147960,u731808469301493760,owner_of_communities_frequent_by
4,u20147960,u731808469301493760,owner_of_communities_frequent_by
5,u20147960,u731808469301493760,owner_of_communities_frequent_by
6,u20147960,u731808469301493760,owner_of_communities_frequent_by
7,u20147960,u731808469301493760,owner_of_communities_frequent_by


# 4) user --mentions--> user

In [8]:
df_mentions_user = df_mentions_base[["source_id", "target_id"]].copy()
df_mentions_user = df_mentions_user.rename(columns={"source_id": "mentioning_tweet_id"})

df_mentions_user["source_id"] = df_mentions_user["mentioning_tweet_id"].map(tweet_to_user_map)
df_mentions_user["relation"] = "mentions"

df_mentions_user = df_mentions_user[
    df_mentions_user["source_id"].notna() &
    (df_mentions_user["source_id"] != df_mentions_user["target_id"])
][["source_id", "target_id", "relation"]]

print(df_mentions_user.shape)
df_mentions_user.head()

(333417, 3)


,source_id,target_id,relation
12573814,u360540433,u1381505906,mentions
12573815,u819756020197167104,u18839785,mentions
12573816,u43963249,u252249233,mentions
12573822,u202982523,u977584704500224001,mentions
12573823,u102708234,u16984977,mentions


# 5) user --replies_to -> user

In [9]:
df_replies_user = df_replies_base[["source_id", "target_id"]].copy()
df_replies_user = df_replies_user.rename(columns={
    "source_id": "reply_tweet_id",
    "target_id": "replied_to_tweet_id"
})

df_replies_user["source_id"] = df_replies_user["reply_tweet_id"].map(tweet_to_user_map)
df_replies_user["target_id"] = df_replies_user["replied_to_tweet_id"].map(tweet_to_user_map)
df_replies_user["relation"] = "replies_to"

df_replies_user = df_replies_user[
    df_replies_user["source_id"].notna() &
    df_replies_user["target_id"].notna() &
    (df_replies_user["source_id"] != df_replies_user["target_id"])
][["source_id", "target_id", "relation"]]

print(df_replies_user.shape)
df_replies_user.head()

(47900, 3)


,source_id,target_id,relation
13644330,u223105455,u19985444,replies_to
13644338,u1157604793,u1329262027,replies_to
13644353,u90570778,u44196397,replies_to
13644375,u1601593543,u38174427,replies_to
13644382,u269873890,u1163992520252153857,replies_to


# 6) user --retweets -> user

In [10]:
df_retweets_user = df_retweets_base[["source_id", "target_id"]].copy()
df_retweets_user = df_retweets_user.rename(columns={
    "source_id": "retweet_tweet_id",
    "target_id": "retweeted_tweet_id"
})

df_retweets_user["source_id"] = df_retweets_user["retweet_tweet_id"].map(tweet_to_user_map)
df_retweets_user["target_id"] = df_retweets_user["retweeted_tweet_id"].map(tweet_to_user_map)
df_retweets_user["relation"] = "retweets"

df_retweets_user = df_retweets_user[
    df_retweets_user["source_id"].notna() &
    df_retweets_user["target_id"].notna() &
    (df_retweets_user["source_id"] != df_retweets_user["target_id"])
][["source_id", "target_id", "relation"]]

print(df_retweets_user.shape)
df_retweets_user.head()

(111247, 3)


,source_id,target_id,relation
13644323,u36876992,u2869101210,retweets
13644341,u2240601558,u6466252,retweets
13644357,u607242831,u177547780,retweets
13644381,u2779511456,u14800270,retweets
13644397,u1271656262,u31656892,retweets


# 7) user --quotes--> user

In [11]:
df_quotes_user = df_quotes_base[["source_id", "target_id"]].copy()
df_quotes_user = df_quotes_user.rename(columns={
    "source_id": "quote_tweet_id",
    "target_id": "quoted_tweet_id"
})

df_quotes_user["source_id"] = df_quotes_user["quote_tweet_id"].map(tweet_to_user_map)
df_quotes_user["target_id"] = df_quotes_user["quoted_tweet_id"].map(tweet_to_user_map)
df_quotes_user["relation"] = "quotes"

df_quotes_user = df_quotes_user[
    df_quotes_user["source_id"].notna() &
    df_quotes_user["target_id"].notna() &
    (df_quotes_user["source_id"] != df_quotes_user["target_id"])
][["source_id", "target_id", "relation"]]

print(df_quotes_user.shape)
df_quotes_user.head()

(19233, 3)


,source_id,target_id,relation
13644520,u43996441,u275686563,quotes
13644574,u993534145,u135647562,quotes
13644578,u72913875,u821102114,quotes
13644594,u18587457,u14552720,quotes
13644799,u162293874,u232294292,quotes


# 8) user --member_of_communities_owned_by --> user

In [12]:
df_membership_to_owner = (
    df_membership_base[["source_id", "target_id"]]
    .rename(columns={"source_id": "list_id", "target_id": "source_id"})
    .merge(
        df_own_base[["source_id", "target_id"]]
        .rename(columns={"source_id": "target_id", "target_id": "list_id"}),
        on="list_id",
        how="inner"
    )
    .copy()
)

df_membership_to_owner["relation"] = "member_of_communities_owned_by"

df_membership_to_owner = df_membership_to_owner[
    df_membership_to_owner["source_id"] != df_membership_to_owner["target_id"]
][["source_id", "target_id", "relation"]].drop_duplicates()

print(df_membership_to_owner.shape)
df_membership_to_owner.head()

(10626, 3)


,source_id,target_id,relation
0,u2239670346,u8919762,member_of_communities_owned_by
1,u15823641,u784597005825871876,member_of_communities_owned_by
2,u1123336792021897216,u16621479,member_of_communities_owned_by
4,u1023072078,u76133155,member_of_communities_owned_by
5,u1769551,u11735032,member_of_communities_owned_by


In [ ]:
# 9) user --discusses_same_topics --> user

In [14]:
from itertools import combinations

#Limit the number of user 
MAX_USERS_PER_HASHTAG = 200

df_discuss_pairs = (
    df_discuss_base[["source_id", "target_id"]]
    .dropna()
    .drop_duplicates()
    .rename(columns={"target_id": "hashtag_id"})
    .copy()
)

df_discuss_pairs["source_id"] = df_discuss_pairs["source_id"].astype(str)
df_discuss_pairs["hashtag_id"] = df_discuss_pairs["hashtag_id"].astype(str)

edge_parts = []

for hashtag_id, users in df_discuss_pairs.groupby("hashtag_id", sort=False)["source_id"]:
    users = pd.unique(users)

    if len(users) < 2:
        continue
    if len(users) > MAX_USERS_PER_HASHTAG:
        continue

    part = pd.DataFrame(combinations(users, 2), columns=["source_id", "target_id"])
    edge_parts.append(part)

if edge_parts:
    df_topics_user = pd.concat(edge_parts, ignore_index=True)
else:
    df_topics_user = pd.DataFrame(columns=["source_id", "target_id"])

df_topics_user_rev = df_topics_user.rename(
    columns={"source_id": "target_id", "target_id": "source_id"}
)

df_topics_user = pd.concat([df_topics_user, df_topics_user_rev], ignore_index=True)
df_topics_user["relation"] = "discuess_same_topic"
df_topics_user = df_topics_user[["source_id", "target_id", "relation"]]

print(df_topics_user.shape)
df_topics_user.head()

(130513338, 3)


,source_id,target_id,relation
0,t1296249717257625602,t1272599383264067585,discuess_same_topic
1,t1296249717257625602,t1240788385972748289,discuess_same_topic
2,t1296249717257625602,t1278003983962058752,discuess_same_topic
3,t1296249717257625602,t1239635416527122433,discuess_same_topic
4,t1296249717257625602,t1289213223925768192,discuess_same_topic


# Combine all user_user_edges

In [23]:
df_user_user_edges_all = pd.concat([
    df_follows,
    df_likes_user,
    df_list_owner_to_user,
    df_mentions_user,
    df_replies_user,
    df_retweets_user,
    df_quotes_user,
    df_membership_to_owner,
    df_topics_user
], ignore_index=True)

print(df_user_user_edges_all.shape)
print(df_user_user_edges_all["relation"].value_counts())

# df_user_user_edges_all.head()

(131317169, 3)
relation
discuess_same_topic                 130513338
mentions                               333417
following                              215336
retweets                               111247
replies_to                              47900
owner_of_communities_frequent_by        43159
likes                                   22913
quotes                                  19233
member_of_communities_owned_by          10626
Name: count, dtype: int64


### Collapse duplicate (source, target, relation) rows into one row per unique source and target pair, with `weight` reflecting how many times that interaction occurred.

In [ ]:
df_user_user_edges_all = (
    df_user_user_edges_all
    .groupby(["source_id", "target_id", "relation"], as_index=False)
    .size()
    .rename(columns={"size": "weight"})
)

print(df_user_user_edges_all.shape)
print(df_user_user_edges_all["relation"].value_counts())

In [ ]:
df_user_user_edges_all

,source_id,target_id,relation,weight
0,u100030266,u358253618,member_of_communities_owned_by,1
1,u1000591,l2982205,following,1
2,u1000591,l996794519625654274,following,1
3,u1000591,u101524268,member_of_communities_owned_by,1
4,u1000591,u105161933,member_of_communities_owned_by,1
...,...,...,...,...
430411,u999947328621395968,u984188226826010624,following,1
430412,u999947328621395968,u984188226826010624,mentions,1
430413,u999947328621395968,u998588353384796167,following,1
430414,u999947328621395968,u998588353384796167,mentions,1


# Summary Statistics

In [ ]:
print("=" * 50)
print("GRAPH EDGE SUMMARY")
print("=" * 50)

print(f"\nTotal edges:         {df_user_user_edges_all.shape[0]:,}")
print(f"Unique source users: {df_user_user_edges_all['source_id'].nunique():,}")
print(f"Unique target users: {df_user_user_edges_all['target_id'].nunique():,}")
print(f"Unique users overall:{pd.concat([df_user_user_edges_all['source_id'], df_user_user_edges_all['target_id']]).nunique():,}")

print("\n" + "=" * 50)
print("RELATION TYPE DISTRIBUTION")
print("=" * 50)
relation_stats = (
    df_user_user_edges_all
    .groupby("relation")
    .agg(
        edge_count=("weight", "count"),
        total_weight=("weight", "sum"),
        mean_weight=("weight", "mean"),
        median_weight=("weight", "median"),
        max_weight=("weight", "max"),
    )
    .sort_values("edge_count", ascending=False)
)
relation_stats["edge_pct"] = (relation_stats["edge_count"] / relation_stats["edge_count"].sum() * 100).round(2)
relation_stats["mean_weight"] = relation_stats["mean_weight"].round(2)
print(relation_stats.to_string())

print("\n" + "=" * 50)
print("OVERALL WEIGHT DISTRIBUTION")
print("=" * 50)
print(df_user_user_edges_all["weight"].describe().round(2).to_string())

print("\n" + "=" * 50)
print("WEIGHT DISTRIBUTION BUCKETS")
print("=" * 50)
bins = [0, 1, 2, 5, 10, 50, float("inf")]
labels = ["1", "2", "3-5", "6-10", "11-50", "50+"]
weight_buckets = pd.cut(df_user_user_edges_all["weight"], bins=bins, labels=labels)
print(weight_buckets.value_counts().sort_index().to_string())

print("\n" + "=" * 50)
print("VALID USER EDGE COVERAGE CHECK")
print("=" * 50)

edge_user_nodes = set(
    pd.concat(
        [df_user_user_edges_all["source_id"], df_user_user_edges_all["target_id"]],
        ignore_index=True
    ).unique()
)

valid_users_set = set(valid_users)
missing_valid_users = valid_users_set - edge_user_nodes

print(f"Valid users with >=1 edge:   {len(valid_users_set) - len(missing_valid_users):,}")
print(f"Valid users with no edges:   {len(missing_valid_users):,}")
print(f"All valid users covered?     {len(missing_valid_users) == 0}")

if len(missing_valid_users) > 0:
    print("\nSample valid users with no edges:")
    print(list(sorted(missing_valid_users))[:20])

GRAPH EDGE SUMMARY

Total edges:         430,416
Unique source users: 8,873
Unique target users: 24,324
Unique users overall:25,851

RELATION TYPE DISTRIBUTION
                                  edge_count  total_weight  mean_weight  median_weight  max_weight  edge_pct
relation                                                                                                    
following                             215336        215336         1.00            1.0           1     50.03
mentions                               98907        333417         3.37            1.0         905     22.98
retweets                               44973        111247         2.47            1.0         414     10.45
likes                                  22913         22913         1.00            1.0           1      5.32
replies_to                             20590         47900         2.33            1.0         185      4.78
quotes                                 12930         19233         1.49      

In [ ]:
# import os
# os.makedirs("datasets", exist_ok=True)
# df_user_user_edges_all.to_parquet("datasets/user_user_edges.parquet", index=False)

NameError: name 'df_user_user_edges_all' is not defined

## GNN Codes Starts here:

# Section 2. Graph Neural Networks
1. Constructing the graph structure (**edge_index**, **edge_weight**, **node_features**, ) where the **nodes are represented by user features**, and **the edge are represented by _____ relationships between nodes (users)**. 
2.. Create the pytorch `Data` object.
3. Run `custom_stratified_split` to get training, validation and test masks.
4. Build custom functions for `EarlyStopping`, `train_validate`, `test`.
5. Train and validate on the following GNN models with `Adam Optimizer`, `learning rate = 0.05` and `decay = 0.01`:
   - `Graph Convolutional Network (GCN)`
   - `GraphSAGE`
   - `Graph Attention Network (GAT)`
6. Evaluate model using `accuracy`, `precision`, `recall`, `f1-score` on the test set and plot `confusion matrix`.

In [24]:
import torch
import torch.nn.functional as F
from torch import nn
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data

C:\Users\Timothy Koh\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [30]:
# Create a mapping of user IDs to node indices >> change it up to index
df_edges_user_user = df_user_user_edges_all.copy()
all_users = pd.Index(df_edges_user_user["source_id"]).union(
    pd.Index(df_edges_user_user["target_id"])
)
user_to_idx = {user_id: idx for idx, user_id in enumerate(all_users)}


# Create edge index and edge weight tensors
edge_list = []
edge_weight_dict = {}

for _, row in df_edges_user_user.iterrows():
    source_idx = user_to_idx[row['source_id']]
    target_idx = user_to_idx[row['target_id']]
    idx_pair = (source_idx, target_idx)
    if (idx_pair not in edge_list):
        edge_list.append(idx_pair)
        edge_weight_dict[idx_pair] = 1
    else:
        edge_weight_dict[idx_pair] += 1

edge_weight_list = list(edge_weight_dict.values())

#force it to be contiguous
edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
edge_weight = torch.tensor(edge_weight_list, dtype=torch.float)

#### Run custom_stratified_split with 80% trainset, 10% valset and 10% testset
#### We will be training the data using the train_mask and evaluate it using the val_mask before testing it on the test_mask to get the final performance scores.

In [31]:
def custom_stratified_split(data, train_ratio, val_ratio, test_ratio):

    # Ensure the ratios sum to 1
    assert train_ratio + val_ratio + test_ratio == 1, "Ratios must sum to 1"

    num_nodes = data.num_nodes
    num_classes = int(torch.max(data.y)) + 1

    # Masks initialization
    train_mask = torch.zeros(num_nodes, dtype=torch.bool)
    val_mask = torch.zeros(num_nodes, dtype=torch.bool)
    test_mask = torch.zeros(num_nodes, dtype=torch.bool)

    for class_idx in range(num_classes):
        # Get indices of nodes in the current class
        class_indices = (data.y == class_idx).nonzero(as_tuple=False).view(-1)
        # Shuffle indices
        class_indices = class_indices[torch.randperm(len(class_indices))]

        # Compute split sizes
        num_train_per_class = int(len(class_indices) * train_ratio)
        num_val_per_class = int(len(class_indices) * val_ratio)

        # Assign masks
        train_mask[class_indices[:num_train_per_class]] = True
        val_mask[class_indices[num_train_per_class:num_train_per_class + num_val_per_class]] = True
        test_mask[class_indices[num_train_per_class + num_val_per_class:]] = True

    return train_mask, val_mask, test_mask

###Step 3. Create pytorch geometric Data object

In [32]:
# create a PyTorch Geometric data object

# tweet embedding + top 10 user features
data = Data(x=node_features_tensor.float(), y=label_tensor.long(), edge_index=edge_index, edge_weight=edge_weight)

train_mask, val_mask, test_mask = custom_stratified_split(data, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1)

NameError: name 'node_features_tensor' is not defined

###Step 4. Build custom functions for `EarlyStopping`, `train_validate` and `test`

In [ ]:
class EarlyStopping:
    def __init__(self, patience=3, verbose=False, delta=0, path='checkpoint.pt', trace_func=print):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.Inf
        self.delta = delta
        self.path = path
        self.trace_func = trace_func

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score <= self.best_score + self.delta:
            self.counter += 1
            self.trace_func(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            self.trace_func(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...')
        self.val_loss_min = val_loss

In [34]:
def train(model, data, criterion, optimizer, has_edge_weight=True):
    model.train()
    optimizer.zero_grad()
    if has_edge_weight:
        out = model(data.x, data.edge_index, data.edge_weight)
    else:
        out = model(data.x, data.edge_index)
    loss = criterion(out[train_mask], data.y[train_mask])
    loss.backward()
    optimizer.step()
    return loss

def validate(model, data, criterion, optimizer, has_edge_weight=True):
    model.eval()
    with torch.no_grad():
        if has_edge_weight:
            out = model(data.x, data.edge_index, data.edge_weight)
        else:
            out = model(data.x, data.edge_index)
        loss = criterion(out[val_mask], data.y[val_mask])

        # for BCE Loss
        _, pred = torch.max(out, 1)

        # Convert to CPU and numpy for sklearn compatibility
        true_labels = data.y[val_mask].cpu().numpy()
        pred_labels = pred[val_mask].cpu().numpy()

        # Calculate precision, recall, and accuracy
        precision = precision_score(true_labels, pred_labels, average='binary')
        recall = recall_score(true_labels, pred_labels, average='binary')
        accuracy = accuracy_score(true_labels, pred_labels)
        f1 = f1_score(true_labels, pred_labels)

    return loss, precision, recall, accuracy, f1

# ----------------------------------------------- Functions for training and testing --------------------------------------------------------

def train_validate(model, data, criterion, optimizer, epoch, early_stopping_patience, has_edge_weight=True):
    early_stopping = EarlyStopping(patience=early_stopping_patience, verbose=False)
    for epoch in range(epoch):
        loss = train(model, data, criterion, optimizer, has_edge_weight)
        val_loss, val_precision, val_recall, val_acc, val_f1 = validate(model, data, criterion, optimizer, has_edge_weight)
        print(f'Epoch {epoch+1} | Train Loss: {loss:.4f}, Val Loss: {val_loss:.4f}, Val Precision: {val_precision:.4f}, Val Recall: {val_recall:.4f}, Val Acc: {val_acc:.4f}, Val f1: {val_f1:.4f}')

        # Early stopping
        early_stopping(val_loss, model)
        if early_stopping.early_stop:
            print("Early stopping")
            break

def test(model, data, criterion, optimizer, has_edge_weight=True, show_cm=True, show_df=False):
    model.eval()
    with torch.no_grad():
        if has_edge_weight:
            out = model(data.x, data.edge_index, data.edge_weight)
        else:
            out = model(data.x, data.edge_index)
        test_loss = criterion(out[test_mask], data.y[test_mask].long())
        pred = out.argmax(dim=1)

        # Convert to CPU and numpy for sklearn compatibility
        true_labels = data.y[test_mask].cpu().numpy()
        pred_labels = pred[test_mask].cpu().numpy()

        # Calculate precision, recall, and accuracy
        precision = precision_score(true_labels, pred_labels, average='binary')
        recall = recall_score(true_labels, pred_labels, average='binary')
        accuracy = accuracy_score(true_labels, pred_labels)
        f1 = f1_score(true_labels, pred_labels)


    test_pred, test_labels, test_loss, test_precision, test_recall, test_acc, test_f1 = pred[test_mask], data.y[test_mask], test_loss, precision, recall, accuracy, f1
    print(f'Test Loss: {test_loss:.4f}, Test Precision: {test_precision:.4f}, Test Recall: {test_recall:.4f}, Test Acc: {test_acc:.4f}, Test F1: {test_f1:.4f}')

    # set whether to plot confusion matrix
    if (show_cm):
        cm = confusion_matrix(test_labels.cpu().numpy(), test_pred.cpu().numpy())
        # Plot the confusion matrix
        plt.figure(figsize=(10, 7))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
        plt.xlabel('Predicted labels')
        plt.ylabel('True labels')
        plt.title('Confusion Matrix')
        plt.show()

    # Create a DataFrame with source_user_id, true labels, and predicted labels
    if show_df:
        predicted_id_df = pd.DataFrame({
            'source_user_id': u["source_user_id"][test_mask.cpu().numpy()],
            'true_label': true_labels,
            'predicted_label': pred_labels
        })
        return predicted_id_df

### GNN Model

Some considerations: what sampling scheme? What aggregation method? What research backs up our choice?

#### 5.2 GraphSAGE Model
GraphSAGE introduces a sampling mechanism and multiple types of aggregators (e.g., mean, LSTM, pooling) for combining the features from neighboring nodes. This Instead of aggregating from all neighbors like GCN, GraphSAGE samples a fixed number of neighbors and then aggregates their features. This makes it scalable to large graphs.

GraphSAGE's ability to perform inductive learning and its neighborhood sampling technique make it particularly well-suited for identifying bots within a network like Twitter.

It can also generalize better to unseen nodes not seen in training which aligns with our objective.


Several important hyperparameters:
1. in_channels (int or tuple) – Size of each input sample, or -1 to derive the size from the first input(s) to the forward method. A tuple corresponds to the sizes of source and target dimensionalities.

2. hidden_channels (int) – Size of each hidden sample.

3. num_layers (int) – Number of message passing layers.

4.  out_channels (int, optional) – If not set to None, will apply a final linear transformation to convert hidden node embeddings to output size out_channels. (default: None)

- `Dimension: 64 and 32`
- `Batch Normalization and feed-forward after input layer`
- `1 GraphSAGE layer` with `mean pooling`
- `Batch Normalization`
- `Dropout: 0.2`
- `Feed-forward network with 2 linear layers`

In [ ]:
from torch_geometric.nn import SAGEConv

class GraphSAGE2(nn.Module):
    def __init__(self, num_features, num_classes):
        super(GraphSAGE2, self).__init__()
        self.bn1 = torch.nn.BatchNorm1d(num_features)
        self.linear1 = torch.nn.Linear(num_features, 64)
        self.conv = SAGEConv(64, 64, aggr='mean')
        self.dropout = nn.Dropout(0.2)
        self.bn2 = torch.nn.BatchNorm1d(64)
        self.linear2 = torch.nn.Linear(64, 32)
        self.linear3 = torch.nn.Linear(32, num_classes)


    def forward(self, x, edge_index):
        x = self.bn1(x)
        x = F.leaky_relu(self.linear1(x))
        x = F.leaky_relu(self.conv(x, edge_index))
        x = self.dropout(x)

        x = self.bn2(x)
        x = F.leaky_relu(self.linear2(x))
        x = self.dropout(x)

        x = self.linear3(x)

        return x

In [ ]:
model = GraphSAGE2(num_features=node_features_tensor.size(1), num_classes=2)

decay = 0.05
lr = 0.001

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=decay)

train_validate(model=model, data=data, criterion=criterion, optimizer=optimizer, epoch=200, has_edge_weight=False, early_stopping_patience=10)

In [ ]:
test(model, data, criterion, optimizer, has_edge_weight=False, show_cm=True)

## Summary statistics 
- Validate that the graph has 20k user nodes
- Ensuring total number of nodes from input matches in final graph
- Check that the label vector aligns with node count
- Ensure number of edges matches the edge weights
- Confirm feature dimension per node is as expected
- Count the number of nodes in test, train and validation set and ensure it covers the whole graph 

In [ ]:
num_nodes_from_mapping = len(user_to_idx)
num_nodes_from_x = data.x.shape[0]
num_labels = data.y.shape[0]
num_edges = data.edge_index.shape[1]
num_edge_weights = data.edge_weight.shape[0] if hasattr(data, "edge_weight") and data.edge_weight is not None else 0
num_features = data.x.shape[1]

print("Summary Stastics for created graph")
print(f"Nodes from user_to_idx:   {num_nodes_from_mapping:,}")
print(f"Nodes in data.x:          {num_nodes_from_x:,}")
print(f"Labels in data.y:         {num_labels:,}")
print(f"Edges in data.edge_index: {num_edges:,}")
print(f"Edge weights:             {num_edge_weights:,}")
print(f"Features per node:        {num_features:,}")

print("\nConsistency checks:")
print(f"x matches node mapping:   {num_nodes_from_x == num_nodes_from_mapping}")
print(f"y matches node count:     {num_labels == num_nodes_from_x}")
print(f"edge_weight matches edge_index: {num_edge_weights == num_edges}")
print(f"20k user nodes:       {num_nodes_from_x >= 20000}")

print("\nMask sizes:")
print(f"Train nodes: {int(train_mask.sum()):,}")
print(f"Val nodes:   {int(val_mask.sum()):,}")
print(f"Test nodes:  {int(test_mask.sum()):,}")
print(f"Mask total matches node count: {(train_mask.sum() + val_mask.sum() + test_mask.sum()).item() == num_nodes_from_x}")

## Performance Measure

### ROC Curve

Comments

### Precision-Recall Curve

Comments